In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import openpyxl
from sklearn.metrics import r2_score
import seaborn as sns
from scipy import stats

In [ ]:
#load the data from the CSV file or excel file
file_path = "../data/insurance.csv"
data = pd.read_csv(file_path)

print("Original columns:", data.columns.tolist())

In [ ]:
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = data.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numeric columns detected:", numeric_cols)
print("Categorical columns detected:", categorical_cols)

binary_cols = [col for col in categorical_cols if data[col].nunique() == 2]
multi_cols = [col for col in categorical_cols if data[col].nunique() > 2]

print("Binary categorical columns:", binary_cols)
print("Multi-category columns:", multi_cols)

# Encode binary columns as 1/0
for col in binary_cols:
    unique_vals = list(data[col].unique())
    data[col] = data[col].map({unique_vals[0]: 0, unique_vals[1]: 1})
    print(f"Binary encoded {col}: {unique_vals[0]}=0, {unique_vals[1]}=1")

# One-hot encode multi-category columns
if len(multi_cols) > 0:
    data = pd.get_dummies(data, columns=multi_cols, drop_first=False)
    print("One-hot encoded:", multi_cols)

print("Columns after encoding:", data.columns.tolist())
# Recalculate numeric columns after encoding
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove zero-variance columns
numeric_cols = [col for col in numeric_cols if data[col].var() > 0]

# Remove columns with missing values
numeric_cols = [col for col in numeric_cols if data[col].isna().sum() == 0]

print("Usable numeric columns:", numeric_cols)

if len(numeric_cols) < 2:
    raise ValueError("Not enough usable numeric columns for regression.")
corr_matrix = data[numeric_cols].corr().abs()
corr_matrix_values = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
max_corr = corr_matrix_values.stack().idxmax()

X_col, Y_col = max_corr

print(f"Strongest correlation pair: {X_col} ↔ {Y_col}")
print(f"Using '{X_col}' as X")
print(f"Using '{Y_col}' as Y")

X = data[[X_col]]
Y = data[Y_col]

categorical_cols = data.select_dtypes(include=['object', 'category']).columns.tolist()

binary_cols = [col for col in categorical_cols if data[col].nunique() == 2]
multi_cols = [col for col in categorical_cols if data[col].nunique() > 2]

print("Binary categorical columns:", binary_cols)
print("Multi-category columns:", multi_cols)

# --- Encode binary columns as 1/0 ---
for col in binary_cols:
    unique_vals = list(data[col].unique())
    # Assign 0 to the first category, 1 to the second
    data[col] = data[col].map({unique_vals[0]: 0, unique_vals[1]: 1})
    print(f"Binary encoded {col}: {unique_vals[0]}=0, {unique_vals[1]}=1")

# --- One-hot encode multi-category columns ---
if len(multi_cols) > 0:
    data = pd.get_dummies(data, columns=multi_cols, drop_first=False)
    print("One-hot encoded:", multi_cols)

print("Columns after encoding:", data.columns.tolist())



In [ ]:
from ipywidgets import interact, Dropdown
import ipywidgets as widgets

# Show all columns
columns = list(data.columns)

print("Available columns:", columns)

# Dropdown widgets
x_dropdown = Dropdown(
    options=columns,
    description='X column:',
    value=columns[0]  # default
)

y_dropdown = Dropdown(
    options=columns,
    description='Y column:',
    value=columns[1] if len(columns) > 1 else columns[0]  # default
)

display(x_dropdown, y_dropdown)

# Button to confirm selection
button = widgets.Button(description="Confirm Selection")
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        print(f"Selected X column: {x_dropdown.value}")
        print(f"Selected Y column: {y_dropdown.value}")

button.on_click(on_button_clicked)

display(button, output)


In [ ]:
# === Use dropdown selections ===

x_col = x_dropdown.value
y_col = y_dropdown.value

print(f"Using X column: {x_col}")
print(f"Using Y column: {y_col}")

# Build variables
X = data[[x_col]]
y = data[y_col]

X.head(), y.head()


In [ ]:
# === CORRELATION HEATMAP ===

plt.figure(figsize=(14, 10))
sns.heatmap(
    data[numeric_cols].corr(),
    cmap="coolwarm",
    annot=True,
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap (After Encoding)", fontsize=16)
plt.show()
# Re-train model to ensure X_col matches current dataset
model = LinearRegression()
model.fit(data[[X_col]], data[Y_col])

# Predict using the same column
Y_pred_full = model.predict(data[[X_col]])


In [ ]:

# Extract X and Y
x = data[x_col].values
y = data[y_col].values

# Remove NaNs
mask = ~np.isnan(x) & ~np.isnan(y)
x_clean = x[mask]
y_clean = y[mask]

# Compute slope & intercept
slope, intercept = np.polyfit(x_clean, y_clean, 1)

# Predictions
Y_pred_full = slope * data[x_col] + intercept

# Compute R²
r_value, p_value = stats.pearsonr(x_clean, y_clean)
r_squared = r_value ** 2

# Premium theme
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 7))

# Scatterplot
sns.scatterplot(
    x=data[x_col],
    y=data[y_col],
    color="#1f77b4",
    alpha=0.6,
    s=70,
    edgecolor="white",
    linewidth=0.5
)

# Sort for smooth line
sorted_idx = np.argsort(data[x_col])
X_sorted = data[x_col].iloc[sorted_idx]
Y_sorted = Y_pred_full.iloc[sorted_idx]

# Regression line
sns.lineplot(
    x=X_sorted,
    y=Y_sorted,
    color="#d62728",
    linewidth=2.5,
    label="Regression Line"
)

# Confidence interval shading
# Compute CI using standard error of estimate
residuals = y_clean - (slope * x_clean + intercept)
SE = np.sqrt(np.sum(residuals**2) / (len(x_clean) - 2))

# 95% CI band
ci = 1.96 * SE
plt.fill_between(
    X_sorted,
    Y_sorted - ci,
    Y_sorted + ci,
    color="#d62728",
    alpha=0.15,
    label="95% Confidence Interval"
)

# Title & labels
plt.title(f"Regression Fit: {x_col} vs {y_col}", fontsize=20, weight="bold", pad=15)
plt.xlabel(x_col, fontsize=14)
plt.ylabel(y_col, fontsize=14)

# R² annotation box
plt.text(
    0.05, 0.95,
    f"Slope: {slope:.4f}\nIntercept: {intercept:.4f}\nR²: {r_squared:.4f}",
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment="top",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.8)
)

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# === SIDE-BY-SIDE HISTOGRAMS ===
plt.figure(figsize=(12,5))

# Histogram for X
plt.subplot(1, 2, 1)
plt.hist(data[x_col], bins=10, color="#007acc", alpha=0.7, edgecolor="black")
plt.title(f"Histogram of X ({x_col})", fontsize=14, weight="bold")
plt.xlabel(f"X ({x_col})", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)

# Histogram for Y
plt.subplot(1, 2, 2)
plt.hist(data[y_col], bins=10, color="#ff5733", alpha=0.7, edgecolor="black")
plt.title(f"Histogram of Y ({y_col})", fontsize=14, weight="bold")
plt.xlabel(f"Y ({y_col})", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# === FIT LINEAR REGRESSION MODEL ===
model = LinearRegression()
model.fit(X, Y)

x = data[x_col].values
y = data[y_col].values

# Remove NaNs just in case
mask = ~np.isnan(x) & ~np.isnan(y)
x_clean = x[mask]
y_clean = y[mask]

# Compute slope and intercept with numpy
slope, intercept = np.polyfit(x_clean, y_clean, 1)


Y_pred = model.predict(X)

print("Slope (m):", model.coef_[0])
print("Intercept (b):", model.intercept_)
print("R² Score:", r2_score(Y, Y_pred))

# Pearson correlation between X and Y
correlation = data[x_col].corr(data[y_col])
print(f"Correlation between {x_col} and {y_col}: {correlation:.4f}")

In [ ]:
# === Linear Regression Summary with Correlation ===

# Calculate correlation
correlation = data[x_col].corr(data[y_col])

print("=== Linear Regression Summary ===")
print(f"X Variable: {x_col}")
print(f"Y Variable: {y_col}")
print(f"Slope: {slope:.4f}")
print(f"Intercept: {intercept}")
print(f"Correlation: {correlation:.4f}")
print(f"Equation: y = {slope:.3f} * x + {intercept:.3f}")